# Force closure
- Cartesian Impedance position control + PI force control

> Flow
1. Declare xml: box with contact site (L/R)
2. Get targets: contact position, contact force direction, contact force magnitude
3. Define each controller: position with PD, contact force with PI

#### 0. Generate Scene

In [1]:
import os
import sys
import numpy as np
import time
import mujoco
sys.path.append(os.path.abspath('../'))
from pp_base_mujoco.VIEWER import *
from pp_base_mujoco.UTILS import *
from pp_base_mujoco.SPEC_HELPER import *
from pp_base_mujoco.KINEMATICS import *

In [2]:
spec_helper = MjSpecHelper()
spec_helper.add_robot(
    path='../asset/panda/panda_ee_sphere.xml',
    body_name="base",
    p=(0, 0.7, 0),
    # r=(0, 0, -1.57),
    r=(0, 0, 0),
    prefix="",
    suffix="_right"
)
spec_helper.add_robot(
    path='../asset/panda/panda_ee_sphere.xml',
    body_name="base",
    p=(0, -0.7, 0),
    # r=(0, 0, 1.57),
    r=(0, 0, 0),
    prefix="",
    suffix="_left"
)
spec_helper.add_geom(
    name="box",
    type='box',
    size=(0.15, 0.15, 0.15),
    freejoint = True,
    p=(0.2, 0, 0.15),
    r=(0, 0, 0),
    rgba=(0.3, 0.3, 0.3, 0.5),
    group=1,
    friction=(1.0, 0.05, 0.01),
    condim=4,
    solimp=(0.98, 0.999, 0.001, 0.5, 2),
    solref=(0.01, 1.0),
    mass=0.5
)
spec_helper.add_site(
    name="contact_right",
    size=(0.05,0.05,0.05),
    p=(0, 0.15+0.05, 0),
    r=(0, 0, 0),
    rgba=(0.3, 0.3, 1.0, 1.0),
    group=2,
    body_name="box"
)
spec_helper.add_site(
    name="contact_left",
    size=(0.05,0.05,0.05),
    p=(0, -0.15-0.05, 0),
    r=(0, 0, 0),
    rgba=(0.3, 0.3, 1.0, 1.0),
    group=2,
    body_name="box"
)
spec_helper.add_site(
    name="center",
    size=(0.05,0.05,0.05),
    p=(0, 0, 0),
    r=(0, 0, 0),
    rgba=(0.3, 0.3, 1.0, 1.0),
    group=2,
    body_name="box"
)

model, data = spec_helper.compile()
spec_helper.save_to_xml("../asset/xml/scene_panda_lr.xml")

In [ ]:
model.opt.timestep = 0.001

#### 1. Initialize Scene

In [4]:
joint_names = get_joint_names(model, data)
joint_names_left = [name for name in joint_names if name is not None and "_left" in name]
joint_names_right = [name for name in joint_names if name is not None and "_right" in name]

""" GO TO INITIAL QPOS """
qpos_init = np.array([0, -0.5, 0, -2.5, 0, 2.0, 0.5]) # set initial qpos
apply_qpos_names(model, data, names=joint_names_left, value=qpos_init)
apply_qpos_names(model, data, names=joint_names_right, value=qpos_init)
mujoco.mj_forward(model, data)

In [5]:
viewer = MUJOCOGLVIEWER(model, data)
viewer.view_geom(group=0, show=False)
viewer.view_geom(group=1, show=True)
mujoco.mj_resetData(model, data)
apply_qpos_names(model, data, names=joint_names_left, value=qpos_init)
apply_qpos_names(model, data, names=joint_names_right, value=qpos_init)
mujoco.mj_forward(model, data)

while viewer.is_alive():
    # mujoco.mj_step(model, data)
    mujoco.mj_kinematics(model, data)
    # mujoco.mj_forward(model, data)
    viewer.render()

viewer.close()
del(viewer)

#### 2. Get EE target & IK

In [6]:
# current end effector position rotation 
p_ee_left = get_p(model, data, name="eef_sphere_left", type='geom')
p_ee_right = get_p(model, data, name="eef_sphere_right", type='geom')
print("Current EE position (left):", p_ee_left)
print("Current EE position (right):", p_ee_right)
# rotation 
R_ee_left = get_R(model, data, name="eef_sphere_left", type='geom')
R_ee_right = get_R(model, data, name="eef_sphere_right", type='geom')
print("Current EE rotation (left):", R_ee_left)
print("Current EE rotation (right):", R_ee_right)

Current EE position (left): [ 0.39240442 -0.7         0.45858535]
Current EE position (right): [0.39240442 0.7        0.45858535]
Current EE rotation (left): [[ 9.59410820e-01  2.82012194e-01  1.02892521e-16]
 [ 2.82012194e-01 -9.59410820e-01 -1.78037586e-17]
 [ 9.36953212e-17  4.60980644e-17 -1.00000000e+00]]
Current EE rotation (right): [[ 9.59410820e-01  2.82012194e-01  1.02892521e-16]
 [ 2.82012194e-01 -9.59410820e-01 -1.78037586e-17]
 [ 9.36953212e-17  4.60980644e-17 -1.00000000e+00]]


In [7]:
# contact position 
p_target_contact_left = get_p(model, data, name="contact_left", type='site')
p_target_contact_right = get_p(model, data, name="contact_right", type='site')
print("Contact target position (left):", p_target_contact_left)
print("Contact target position (right):", p_target_contact_right)

R_target_contact_left = [[1.0, 0.0, 0.0],[0.0, 0.0, 1.0],[0.0, -1.0, 0.0]]
R_target_contact_right = [[1.0, 0.0, 0.0],[0.0, 0.0, -1.0],[0.0, 1.0, 0.0]]
print("Target contact frame rotation (left):", R_target_contact_left)
print("Target contact frame rotation (right):", R_target_contact_right)

Contact target position (left): [ 0.2  -0.2   0.15]
Contact target position (right): [0.2  0.2  0.15]
Target contact frame rotation (left): [[1.0, 0.0, 0.0], [0.0, 0.0, 1.0], [0.0, -1.0, 0.0]]
Target contact frame rotation (right): [[1.0, 0.0, 0.0], [0.0, 0.0, -1.0], [0.0, 1.0, 0.0]]


In [8]:
# get error 
pos_error_left, rotvec_error_left = get_ik_error_clipped(
    p_target=p_target_contact_left,
    r_target=R_target_contact_left,
    p_current=p_ee_left,
    r_current=R_ee_left,)
print("Position error (left):", pos_error_left)
print("Rotation error (left):", rotvec_error_left)

pos_error_right, rotvec_error_right = get_ik_error_clipped(
    p_target=p_target_contact_right,
    r_target=R_target_contact_right,
    p_current=p_ee_right,
    r_current=R_ee_right,)
print("Position error (right):", pos_error_right)
print("Rotation error (right):", rotvec_error_right)

Position error (left): [-0.02  0.02 -0.02]
Rotation error (left): [ 0.2         0.12790796 -0.12790796]
Position error (right): [-0.02 -0.02 -0.02]
Rotation error (right): [-0.2        -0.12790796 -0.12790796]


In [9]:
def get_jacobian_franka_ee_bimanual():
    jacobian_p_right, jacobian_r_right = get_jacobian(model, data, 'eef_sphere_right', type='geom', joints_use=joint_names_right)
    # # reduce jacobian of 3x14 to 3x7, first half for right arm 
    # jacobian_p_right = jacobian_p_right[:, :7]
    # jacobian_r_right = jacobian_r_right[:, :7]
    jacobian_p_left, jacobian_r_left = get_jacobian(model, data, 'eef_sphere_left', type='geom', joints_use=joint_names_left)
    # # reduce jacobian of 3x14 to 3x7, second half for left arm
    # jacobian_p_left = jacobian_p_left[:, 7:]
    # jacobian_r_left = jacobian_r_left[:, 7:]

    return jacobian_p_left, jacobian_r_left, jacobian_p_right, jacobian_r_right

#### 3. Reach target EE pR with IK

In [10]:
viewer = MUJOCOGLVIEWER(model, data)
viewer.view_geom(group=0, show=False)
viewer.view_geom(group=1, show=True)
mujoco.mj_resetData(model, data)
apply_qpos_names(model, data, names=joint_names_left, value=qpos_init)
apply_qpos_names(model, data, names=joint_names_right, value=qpos_init)
mujoco.mj_forward(model, data)

while viewer.is_alive():
    # get current EE position & rotation
    p_ee_left = get_p(model, data, name="eef_sphere_left", type='geom')
    p_ee_right = get_p(model, data, name="eef_sphere_right", type='geom')
    R_ee_left = get_R(model, data, name="eef_sphere_left", type='geom')
    R_ee_right = get_R(model, data, name="eef_sphere_right", type='geom')
    # get target p (R is globally fixed )
    p_target_contact_left = get_p(model, data, name="contact_left", type='site')
    p_target_contact_right = get_p(model, data, name="contact_right", type='site')
    # calculate ik error 
    pos_error_left, rotvec_error_left = get_ik_error_clipped(
        p_target=p_target_contact_left,
        r_target=R_target_contact_left,
        p_current=p_ee_left,
        r_current=R_ee_left,
        )
    pos_error_right, rotvec_error_right = get_ik_error_clipped(
        p_target=p_target_contact_right,
        r_target=R_target_contact_right,
        p_current=p_ee_right,
        r_current=R_ee_right,
        )
    error_left = np.concatenate([pos_error_left, rotvec_error_left])
    error_right = np.concatenate([pos_error_right, rotvec_error_right])

    # terminate condition
    if np.linalg.norm(pos_error_left) < 0.01 and np.linalg.norm(rotvec_error_left) < 0.01:
        qpos_left = get_qpos_with_names(model, data, names=joint_names_left)
        # print("\r left arm Target reached!", end="")
    if np.linalg.norm(rotvec_error_right) < 0.01 and np.linalg.norm(pos_error_right) < 0.01:
        qpos_right = get_qpos_with_names(model, data, names=joint_names_right)
        print("\r right arm Target reached!", end="")

    # calculate jacobian
    jac_p_left, jac_r_left, jac_p_right, jac_r_right = get_jacobian_franka_ee_bimanual() 
    jac_left = np.concatenate([jac_p_left, jac_r_left], axis=0)
    jac_right = np.concatenate([jac_p_right, jac_r_right], axis=0)
    # jac_left_inverse = get_pseudo_inverse(jac_left, method="dls", damping=0.1)
    # jac_right_inverse = get_pseudo_inverse(jac_right, method="dls", damping=0.1)
    jac_left_inverse = get_pseudo_inverse(jac_left, method="svd", sigma_threshold=1e-3)
    jac_right_inverse = get_pseudo_inverse(jac_right, method="svd", sigma_threshold=1e-3)
    # calculate qpos error
    qpos_error_left = jac_left_inverse @ error_left
    qpos_error_right = jac_right_inverse @ error_right
    print("qpos error (left):", qpos_error_left)
    print("qpos error (right):", qpos_error_right)
    q_left_updated = get_qpos_with_names(model, data, names=joint_names_left) + qpos_error_left
    q_right_updated = get_qpos_with_names(model, data, names=joint_names_right) + qpos_error_right
    # update qpos
    apply_qpos_names(model, data, names=joint_names_left, value=q_left_updated)
    apply_qpos_names(model, data, names=joint_names_right, value=q_right_updated)
    mujoco.mj_forward(model, data)
    viewer.render()
    time.sleep(0.05) # for visualization stability

# close
viewer.close()
del(viewer)

qpos error (left): [ 0.04685293 -0.01484621 -0.02665555 -0.0345848   0.20589594 -0.10816937
  0.0656855 ]
qpos error (right): [-0.09430104 -0.08922464  0.06932736 -0.15232313 -0.1833973   0.19100645
  0.17076761]
qpos error (left): [ 0.04417435 -0.04787266 -0.02876635 -0.09583265  0.21467108  0.03795719
  0.03983966]
qpos error (right): [-0.12831477 -0.05365836  0.09170451 -0.09528286 -0.19644864  0.05328717
  0.17755726]
qpos error (left): [ 0.06127659 -0.05971016 -0.03219337 -0.11647123  0.2188857   0.08932217
 -0.03774093]
qpos error (right): [-0.1225975  -0.04150525  0.08786313 -0.07274654 -0.19085629 -0.00133566
  0.12762468]
qpos error (left): [ 0.08301651 -0.04929594 -0.04110954 -0.10555619  0.22469738  0.07356384
 -0.10043238]
qpos error (right): [-0.11399091 -0.03839941  0.07884139 -0.07304466 -0.19720717  0.01034093
  0.10405106]
qpos error (left): [ 0.09630791 -0.03181386 -0.05006571 -0.0870179   0.2306629   0.03782215
 -0.13857271]
qpos error (right): [-0.11764899 -0.035529

In [11]:
qpos_left_saved = qpos_left 
qpos_right_saved = qpos_right

#### 4-1. End effector position: Cartesian Impedance controller
- Declare gains & jacobians
- position targets & differences

In [12]:
# Gains
Kp_ee = 100.0
Kd_ee = 20.0
Kp_force = 10.0
Ki_force = 1.0
# get jacobian transpose
jac_p_left, jac_R_left, jac_p_right, jac_R_Right = get_jacobian_franka_ee_bimanual()
jac_left = np.concatenate([jac_p_left, jac_R_left], axis=0)
jac_right = np.concatenate([jac_p_right, jac_R_Right], axis=0)
# pseudo inverse of positional jacobian
jac_left_inverse = get_pseudo_inverse(jac_left, method="svd", sigma_threshold=1e-3)
jac_right_inverse = get_pseudo_inverse(jac_right, method="svd", sigma_threshold=1e-3)

In [13]:
# get positional error
p_ee_target_left = get_p(model, data, name="contact_left", type='site')
p_ee_target_right = get_p(model, data, name="contact_right", type='site')
p_ee_left = get_p(model, data, name="eef_sphere_left", type='geom')
p_ee_right = get_p(model, data, name="eef_sphere_right", type='geom')
p_ee_left_error = p_ee_target_left - p_ee_left
p_ee_right_error = p_ee_target_right - p_ee_right

v_ee_target_left = np.zeros(3)
v_ee_target_right = np.zeros(3)
qvel_left = get_qvel_with_names(model, data, names=joint_names_left)
qvel_right = get_qvel_with_names(model, data, names=joint_names_right)
v_ee_left = jac_p_left @ qvel_left
v_ee_right = jac_p_right @ qvel_right
v_ee_left_error = v_ee_target_left - v_ee_left
v_ee_right_error = v_ee_target_right - v_ee_right

# calculate torque 
f_ee_desired_left = Kp_ee * p_ee_left_error + Kd_ee * v_ee_left_error
f_ee_desired_right = Kp_ee * p_ee_right_error + Kd_ee * v_ee_right_error
torque_left = jac_p_left.T @ f_ee_desired_left
torque_right = jac_p_right.T @ f_ee_desired_right

#### 4-2. Push force: PI controller for cartesian force
- Force direction target with site positions
- Force magnitude: calculate with friction

In [14]:
def get_body_contact_force_position(
        model,
        data,
        body1_name,
        body2_name,
        ):
    body1_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, body1_name)
    body2_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, body2_name)
    contact_forces = []
    contact_positions = []
    for i in range(data.ncon):
        contact = data.contact[i]
        body1_in_contact_id = model.geom_bodyid[contact.geom1]
        body2_in_contact_id = model.geom_bodyid[contact.geom2]
        if (body1_in_contact_id == body1_id and body2_in_contact_id == body2_id) or (body1_in_contact_id == body2_id and body2_in_contact_id == body1_id):
            force = np.zeros(6) # (6,)
            mujoco.mj_contactForce(model,data,i,force)
            contact_forces.append(force)
            contact_positions.append(contact.pos)
    return contact_forces, contact_positions

In [15]:
# gravity compensation force: upper direction 
box_mass = model.body_mass[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "box")]
print("Box mass:", box_mass)
gravity = model.opt.gravity
print("Gravity:", gravity)
f_gravity_compensation = np.array([0, 0, box_mass * gravity[2]])
print("Gravity compensation force:", f_gravity_compensation)

Box mass: 0.5
Gravity: [ 0.    0.   -9.81]
Gravity compensation force: [ 0.     0.    -4.905]


In [16]:
# get desired cartesian force direction & magnitude 
p_center = get_p(model, data, name="center", type='site')
direction_push_left = (p_center - p_target_contact_left) / np.linalg.norm(p_center - p_target_contact_left)
direction_push_right = (p_center - p_target_contact_right) / np.linalg.norm(p_center - p_target_contact_right)
force_magnitude = 5.0 # 5 newtons 
f_push_target_left = Kp_force * (force_magnitude * direction_push_left)
f_push_target_right = Kp_force * (force_magnitude * direction_push_right)
# gravity compensation force: upper direction 
box_mass = model.body_mass[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "box")]
gravity = model.opt.gravity
f_gravity_compensation = np.array([0, 0, box_mass * gravity[2]])
print("Gravity compensation force:", f_gravity_compensation)
f_push_target_left += f_gravity_compensation/2
f_push_target_right += f_gravity_compensation/2
# get current contact force
# f_push_left, p_contact_left = get_body_contact_force_position(model, data, body1_name="eef_sphere_left", body2_name="box")
# f_push_right, p_contact_right = get_body_contact_force_position(model, data, body1_name="eef_sphere_right", body2_name="box")
f_push_left, p_contact_left = get_body_contact_force_position(model, data, body1_name="right_hand_left", body2_name="box")
f_push_right, p_contact_right = get_body_contact_force_position(model, data, body1_name="right_hand_right", body2_name="box")
print("Current contact force (left):", f_push_left, "Contact position (left):", p_contact_left)
print("Current contact force (right):", f_push_right, "Contact position (right):", p_contact_right)

# process contact 
if len(f_push_left) == 0:
    f_push_left = np.zeros(3)
else:
    f_push_left = f_push_left[0][:3] # take the first contact and only force part
if len(f_push_right) == 0:
    f_push_right = np.zeros(3)
else:
    f_push_right = f_push_right[0][:3] # take the first contact and only force part

# desired force with PI controller
f_accumulated_left = np.zeros(3)
f_accumulated_right = np.zeros(3)
f_push_error_left = f_push_target_left - f_push_left
f_push_error_right = f_push_target_right - f_push_right
f_accumulated_left += f_push_error_left * 0.01 # integral term with dt=0.01s
f_accumulated_right += f_push_error_right * 0.01

# get torque
f_push_desired_left = f_push_target_left + Kp_force * f_push_error_left + Ki_force * f_accumulated_left
f_push_desired_right = f_push_target_right + Kp_force * f_push_error_right + Ki_force * f_accumulated_right
torque_left = jac_p_left.T @ f_push_desired_left
torque_right = jac_p_right.T @ f_push_desired_right

Gravity compensation force: [ 0.     0.    -4.905]
Current contact force (left): [] Contact position (left): []
Current contact force (right): [array([0.44483133, 0.44483133, 0.        , 0.        , 0.        ,
       0.        ])] Contact position (right): [array([0.2 , 0.15, 0.15])]


#### 5. Iterate
- Iteratively apply two torques

In [17]:
actuator_names = get_actuator_names(model, data)
print("Actuator names:", actuator_names)
actuator_names_left = [name for name in actuator_names if name is not None and "_left" in name]
print("Left actuator names:", actuator_names_left)
actuator_names_right = [name for name in actuator_names if name is not None and "_right" in name]
print("Right actuator names:", actuator_names_right)

Actuator names: ['torq_j1_right', 'torq_j2_right', 'torq_j3_right', 'torq_j4_right', 'torq_j5_right', 'torq_j6_right', 'torq_j7_right', 'torq_j1_left', 'torq_j2_left', 'torq_j3_left', 'torq_j4_left', 'torq_j5_left', 'torq_j6_left', 'torq_j7_left']
Left actuator names: ['torq_j1_left', 'torq_j2_left', 'torq_j3_left', 'torq_j4_left', 'torq_j5_left', 'torq_j6_left', 'torq_j7_left']
Right actuator names: ['torq_j1_right', 'torq_j2_right', 'torq_j3_right', 'torq_j4_right', 'torq_j5_right', 'torq_j6_right', 'torq_j7_right']


In [18]:
print("bias force:", data.qfrc_bias)

bias force: [ 0.00000000e+00 -2.80489575e+01  4.38529168e+00  9.88639500e+00
 -1.46612425e-01 -8.71212897e-01 -2.22044605e-16  0.00000000e+00
 -2.93078596e+01  2.38076819e-01  1.02619278e+01  4.77821551e-02
 -9.04496860e-01  0.00000000e+00  0.00000000e+00  0.00000000e+00
  4.90500000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]


In [19]:
Kp_ee = 1000.0
Kd_ee = 500.0
Kp_force = 30.0
Ki_force = 1.0
Kp_qpos = 10.0
Kd_qpos = 2.0
Ki_qpos = 0.1
force_magnitude = 10.0 # 5 newtons 

viewer = MUJOCOGLVIEWER(model, data)
viewer.view_geom(group=0, show=False)
viewer.view_geom(group=1, show=True)
viewer.options[0].flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = True
render_tick = 0
mujoco.mj_resetData(model, data)
apply_qpos_names(model, data, names=joint_names_left, value=qpos_left_saved)
apply_qpos_names(model, data, names=joint_names_right, value=qpos_right_saved)
mujoco.mj_forward(model, data)

floor_geom_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_GEOM, "floor")
model.geom_conaffinity[floor_geom_id] = 1
model.geom_contype[floor_geom_id] = 1

f_push_left = np.zeros(3)
f_push_right = np.zeros(3)
p_ee_target_left = get_p(model, data, name="contact_left", type='site').copy() + np.array([0.1, 0, 0])
p_ee_target_right = get_p(model, data, name="contact_right", type='site').copy() + np.array([0.1, 0, 0])

start_time = time.time()
while viewer.is_alive():
    # key input 
    glfw.poll_events()
    if glfw.get_key(viewer.windows[0], glfw.KEY_W) == glfw.PRESS:
        # move upward 
        p_ee_target_left += np.array([0, 0, 0.001])
        p_ee_target_right += np.array([0, 0, 0.001])
    if glfw.get_key(viewer.windows[0], glfw.KEY_S) == glfw.PRESS:
        # move downward 
        p_ee_target_left += np.array([0, 0, -0.001])
        p_ee_target_right += np.array([0, 0, -0.001])
    if glfw.get_key(viewer.windows[0], glfw.KEY_A) == glfw.PRESS:
        # move left 
        p_ee_target_left += np.array([0, -0.001, 0])
        p_ee_target_right += np.array([0, -0.001, 0])
    if glfw.get_key(viewer.windows[0], glfw.KEY_D) == glfw.PRESS:
        # move right
        p_ee_target_left += np.array([0, 0.001, 0])
        p_ee_target_right += np.array([0, 0.001, 0])

    if time.time() - start_time > 1: # run for 20 seconds
        # turn off contype of floor 
        floor_geom_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_GEOM, "floor")
        model.geom_conaffinity[floor_geom_id] = 0
        model.geom_contype[floor_geom_id] = 0
    # get jacobian transpose
    jac_p_left, jac_R_left, jac_p_right, jac_R_Right = get_jacobian_franka_ee_bimanual()
    jac_left = np.concatenate([jac_p_left, jac_R_left], axis=0)
    jac_right = np.concatenate([jac_p_right, jac_R_Right], axis=0)
    jac_left_inverse = get_pseudo_inverse(jac_left, method="svd", sigma_threshold=1e-3)
    jac_right_inverse = get_pseudo_inverse(jac_right, method="svd", sigma_threshold=1e-3)
    
    # pd control torque 
    # p_ee_target_left = get_p(model, data, name="contact_left", type='site')
    # p_ee_target_right = get_p(model, data, name="contact_right", type='site')
    p_ee_left = get_p(model, data, name="eef_sphere_left", type='geom')
    p_ee_right = get_p(model, data, name="eef_sphere_right", type='geom')
    p_ee_left_error = p_ee_target_left - p_ee_left
    p_ee_right_error = p_ee_target_right - p_ee_right
    v_ee_target_left = np.zeros(3)
    v_ee_target_right = np.zeros(3)
    qvel_left = get_qvel_with_names(model, data, names=joint_names_left)
    qvel_right = get_qvel_with_names(model, data, names=joint_names_right)
    v_ee_left = jac_p_left @ qvel_left
    v_ee_right = jac_p_right @ qvel_right
    v_ee_left_error = v_ee_target_left - v_ee_left
    v_ee_right_error = v_ee_target_right - v_ee_right
    f_ee_desired_left = Kp_ee * p_ee_left_error + Kd_ee * v_ee_left_error
    f_ee_desired_right = Kp_ee * p_ee_right_error + Kd_ee * v_ee_right_error
    torque_ee_left = jac_p_left.T @ f_ee_desired_left
    torque_ee_right = jac_p_right.T @ f_ee_desired_right
    print(f"norm of end effector torque left: {np.linalg.norm(torque_ee_left):.2f}")
    print(f"norm of end effector torque right: {np.linalg.norm(torque_ee_right):.2f}")

    # get push force torque 
    p_center = get_p(model, data, name="center", type='site')
    direction_push_left = (p_center - p_target_contact_left) / np.linalg.norm(p_center - p_target_contact_left)
    direction_push_right = (p_center - p_target_contact_right) / np.linalg.norm(p_center - p_target_contact_right)
    f_push_target_left = force_magnitude * direction_push_left
    f_push_target_right = force_magnitude * direction_push_right
    # gravity compensation 
    box_mass = model.body_mass[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "box")]
    gravity = model.opt.gravity
    f_gravity_compensation = np.array([0, 0, box_mass * gravity[2]])
    f_push_target_left += f_gravity_compensation
    f_push_target_right += f_gravity_compensation
    f_push_left, p_contact_left = get_body_contact_force_position(model, data, body1_name="right_hand_left", body2_name="box")
    f_push_right, p_contact_right = get_body_contact_force_position(model, data, body1_name="right_hand_right", body2_name="box")
    if len(f_push_left) == 0:
        f_push_left = np.zeros(3)
    else:
        f_push_left = f_push_left[0][:3] # take the first contact and only force part
    if len(f_push_right) == 0:
        f_push_right = np.zeros(3)
    else:
        f_push_right = f_push_right[0][:3] # take the first contact and only force part
    
    f_accumulated_left = np.zeros(3)
    f_accumulated_right = np.zeros(3)
    f_push_error_left = f_push_target_left - f_push_left
    f_push_error_right = f_push_target_right - f_push_right
    f_accumulated_left += f_push_error_left * 0.01 # integral term with dt=0.01s
    f_accumulated_right += f_push_error_right * 0.01
    f_push_desired_left = f_push_target_left + Kp_force * f_push_error_left + Ki_force * f_accumulated_left
    f_push_desired_right = f_push_target_right + Kp_force * f_push_error_right + Ki_force * f_accumulated_right
    torque_push_left = jac_p_left.T @ f_push_desired_left
    torque_push_right = jac_p_right.T @ f_push_desired_right
    print(f"norm of push torque left: {np.linalg.norm(torque_push_left):.2f}")
    print(f"norm of push torque right: {np.linalg.norm(torque_push_right):.2f}")
    # print(f"left arm force: {f_push_left}, right arm force: {f_push_right}")

    # q difference PID 
    q_target_left = qpos_left_saved
    q_target_right = qpos_right_saved
    q_current_left = get_qpos_with_names(model, data, names=joint_names_left)
    q_current_right = get_qpos_with_names(model, data, names=joint_names_right)
    q_error_left = q_target_left - q_current_left
    q_error_right = q_target_right - q_current_right
    q_error_derivative_left = -get_qvel_with_names(model, data, names=joint_names_left)
    q_error_derivative_right = -get_qvel_with_names(model, data, names=joint_names_right)
    torque_qpos_left = Kp_qpos * q_error_left + Kd_qpos * q_error_derivative_left
    torque_qpos_right = Kp_qpos * q_error_right + Kd_qpos * q_error_derivative_right
    print(f"norm of qpos torque left: {np.linalg.norm(torque_qpos_left):.2f}")
    print(f"norm of qpos torque right: {np.linalg.norm(torque_qpos_right):.2f}")
    torque_qpos_left = torque_qpos_left * 10
    torque_qpos_right = torque_qpos_right * 10

    # bias force (use dof address index)
    joint_dofadr_left = [model.jnt_dofadr[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, name)] for name in joint_names_left]
    joint_dofadr_right = [model.jnt_dofadr[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, name)] for name in joint_names_right]
    q_frc_bias_left = data.qfrc_bias[joint_dofadr_left]
    q_frc_bias_right = data.qfrc_bias[joint_dofadr_right]

    # total torque
    torque_ee_left = torque_ee_left * 20.0 # * 0.1
    torque_ee_right = torque_ee_right * 20.0
    # torque_push_left = torque_push_left * 0.1 
    # torque_push_right = torque_push_right * 0.1
    calculated_torque_left = (torque_ee_left + torque_push_left + torque_qpos_left) * 0.1 # *1e-1
    calculated_torque_right = (torque_ee_right + torque_push_right + torque_qpos_right) * 0.1
    total_torque_left = calculated_torque_left + q_frc_bias_left
    total_torque_right = calculated_torque_right + q_frc_bias_right
    # total_torque_left = q_frc_bias_left
    # total_torque_right = q_frc_bias_right
    apply_ctrl_names(model, data, names=actuator_names_left, value = total_torque_left)
    apply_ctrl_names(model, data, names=actuator_names_right, value = total_torque_right)

    mujoco.mj_step(model, data)
    if render_tick % 10 == 0:
        viewer.render()
    render_tick += 1
    # time.sleep(0.1)


viewer.close()
del(viewer)

norm of end effector torque left: 74.99
norm of end effector torque right: 73.34
norm of push torque left: 117.39
norm of push torque right: 137.64
norm of qpos torque left: 0.00
norm of qpos torque right: 0.00
norm of end effector torque left: 62.61
norm of end effector torque right: 62.99
norm of push torque left: 117.39
norm of push torque right: 248.37
norm of qpos torque left: 0.19
norm of qpos torque right: 0.16
norm of end effector torque left: 51.43
norm of end effector torque right: 52.64
norm of push torque left: 525.53
norm of push torque right: 125.18
norm of qpos torque left: 0.33
norm of qpos torque right: 0.31
norm of end effector torque left: 46.09
norm of end effector torque right: 43.35
norm of push torque left: 762.92
norm of push torque right: 673.78
norm of qpos torque left: 0.41
norm of qpos torque right: 0.43
norm of end effector torque left: 43.72
norm of end effector torque right: 40.84
norm of push torque left: 661.67
norm of push torque right: 667.23
norm of 

2026-04-09 21:06:34.901 python[26031:6346000] TSM AdjustCapsLockLEDForKeyTransitionHandling - _ISSetPhysicalKeyboardCapsLockLED Inhibit



norm of end effector torque right: 36.45
norm of push torque left: 744.91
norm of push torque right: 797.68
norm of qpos torque left: 1.05
norm of qpos torque right: 1.30
norm of end effector torque left: 36.07
norm of end effector torque right: 36.37
norm of push torque left: 737.41
norm of push torque right: 789.43
norm of qpos torque left: 1.04
norm of qpos torque right: 1.29
norm of end effector torque left: 35.87
norm of end effector torque right: 36.24
norm of push torque left: 729.98
norm of push torque right: 780.62
norm of qpos torque left: 1.04
norm of qpos torque right: 1.28
norm of end effector torque left: 35.63
norm of end effector torque right: 36.05
norm of push torque left: 723.17
norm of push torque right: 772.83
norm of qpos torque left: 1.03
norm of qpos torque right: 1.28
norm of end effector torque left: 35.37
norm of end effector torque right: 35.83
norm of push torque left: 717.59
norm of push torque right: 766.40
norm of qpos torque left: 1.03
norm of qpos tor